# DueCare — launch the workbench in a notebook

Runs the DueCare FastAPI workbench and exposes it at a public URL. Works on
Colab, Kaggle (Internet on), or local Jupyter. Do not paste real worker data
into a public tunnel — use the local/Docker deployment for anything sensitive.

## 1. Install DueCare from source

In [ ]:
import subprocess, sys, os
REPO = os.environ.get("DUECARE_REPO", "https://github.com/TaylorAmarelTech/gemma4_comp")
REF  = os.environ.get("DUECARE_COMMIT_SHA", "master")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"], check=True)
# Install the DueCare workspace packages from the checkout (clone if needed).
if not os.path.isdir("gemma4_comp"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO, "gemma4_comp"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                "gemma4_comp/packages/duecare-llm-chat"], check=False)
print("installed (see output above for any resolver notes)")

## 2. Start Ollama + pull a small Gemma 4 (optional; skipped if unavailable)

In [ ]:
import shutil, subprocess, time
if shutil.which("ollama"):
    subprocess.Popen(["ollama", "serve"])
    time.sleep(5)
    subprocess.run(["ollama", "pull", "gemma4:e2b"], check=False)
    print("ollama ready; gemma4:e2b pulled")
else:
    print("ollama not found — the server still starts; load a model from the UI or an API call")

## 3. Launch the DueCare server in the background

In [ ]:
import subprocess, sys, time, urllib.request
PORT = 8080
server = subprocess.Popen([sys.executable, "-m", "duecare.chat.run_server",
                           "--host", "0.0.0.0", "--port", str(PORT)])
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2); break
    except Exception:
        time.sleep(2)
print(f"server process pid={server.pid} on port {PORT}")

## 4. Expose a public URL (cloudflared, then localtunnel fallback)

In [ ]:
import subprocess, shutil, sys, re, time
PORT = 8080
def try_cloudflared():
    if not shutil.which("cloudflared"):
        subprocess.run(["bash","-lc","curl -L --silent --output /usr/local/bin/cloudflared "
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
                        "&& chmod +x /usr/local/bin/cloudflared"], check=False)
    if shutil.which("cloudflared"):
        p = subprocess.Popen(["cloudflared","tunnel","--url",f"http://127.0.0.1:{PORT}"],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for _ in range(60):
            line = p.stdout.readline()
            m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
            if m: return m.group(0)
            time.sleep(1)
    return None
url = try_cloudflared()
print("OPEN THIS URL:", url or "tunnel unavailable — use the local http://127.0.0.1:8080 if running locally")

## Done

Open the printed URL to reach the DueCare workbench: chat with the harness,
compare baseline vs harnessed responses, run bulk file review, and inspect the
activity log. Stop this notebook to take the public URL down.